# geo0265 – Height benchmarks in NRW: from CSV to GeoPackage

**Unit:** geo0260 GNSS, RTK and SAPOS · **Slides:** `geo0260_GNSS` (sections *SAPOS* and *Heights: from GNSS to NHN*) · **Data:** downloaded by this notebook from [opengeodata.nrw.de](https://www.opengeodata.nrw.de/produkte/geobasis/rb/fd/)

Geobasis NRW publishes the official **height benchmarks** (*Höhenfestpunkte*, HFP) of North Rhine-Westphalia as open data: about 88 000 metal bolts on buildings, walls, bridges and pillars, each with a levelled height above NHN (DHHN2016). The file is a plain CSV with an east and a north coordinate (*Rechtswert* and *Hochwert* in the old German terminology), and nothing in the file says which coordinate reference system they belong to.

In this notebook you
1. read a CSV that is not in the format pandas expects (no header, `;`, decimal comma, IDs with digits only),
2. find out **which EPSG code** the coordinates have, from the numbers themselves, and compare your answer with the **metadata**,
3. save the benchmarks as a **GeoPackage** that QGIS opens without questions,
4. find the benchmarks **next to the Green Fab Lab and the campus Kamp-Lintfort**, the places where we measure with RTK,
5. (stretch) look at the **height history** of these benchmarks: Kamp-Lintfort is a coal-mining town.

## 0. Setup

In [ ]:
from pathlib import Path
import re
import xml.etree.ElementTree as ET

import requests
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from pyproj import CRS, Transformer
from shapely.geometry import Point

pd.set_option("display.width", 160)
pd.set_option("display.float_format", "{:.3f}".format)   # no 3.2e+07 for coordinates

BASE = "https://www.opengeodata.nrw.de/produkte/geobasis/rb/fd"
URL_PL = f"{BASE}/hfp_pl_csv/hfp_pl.csv"                 # point list (one row per benchmark)
URL_ZF = f"{BASE}/hfp_plzf_csv/hfp_plzf.csv"             # point list with time series (stretch §6)
URL_FORMAT = f"{BASE}/hfp_pl_csv/Datenformatbeschreibung.pdf"
URL_USERINFO = f"{BASE}/hfp_pl_csv/Nutzerinformation.pdf"
# ISO 19139 metadata record of the data set (GDI-DE catalogue, record 74199fc7-…)
META_ID = "74199fc7-c6d6-40ce-b5da-b43cbc1139ac"
URL_META = ("https://gdk.gdi-de.org/gdi-de/srv/eng/csw?REQUEST=GetRecordById&SERVICE=CSW&VERSION=2.0.2"
            "&OUTPUTSCHEMA=http://www.isotc211.org/2005/gmd&ELEMENTSETNAME=full&ID=" + META_ID)
URL_META_HTML = f"https://ckan.open.nrw.de/dataset/{META_ID}"   # same record, human-readable (Open.NRW)

ORIG = Path("data") / "original" / "geo0265"      # downloads (not in git)
OUT = Path("data") / "derived" / "geo0265"        # results (not in git)
ORIG.mkdir(parents=True, exist_ok=True)
OUT.mkdir(parents=True, exist_ok=True)

# Places where we measure with RTK, (λ, φ) = (lon, lat) in ETRS89 / WGS84 degrees
PLACES = {
    "Green Fab Lab": (6.548, 51.493),          # centre of the drone flight area
    "Campus Kamp-Lintfort": (6.5457, 51.4991), # main building, Friedrich-Heinrich-Allee 25
}

## 1. Download

The data set lives in the folder *Raumbezug → Festpunktdaten* of opengeodata.nrw.de. Open [the folder](https://www.opengeodata.nrw.de/produkte/geobasis/rb/fd/) in a browser: every folder answers with a small JSON listing (files, sizes, dates) and contains the format description as PDF.

The function below downloads a file only once and keeps it in `data/original/geo0265/`.

In [ ]:
def download(url, folder=ORIG):
    """Download url into folder unless the file is already there; return the local path."""
    path = folder / url.rsplit("/", 1)[-1]
    if not path.exists():
        r = requests.get(url, timeout=120)
        r.raise_for_status()
        path.write_bytes(r.content)
    print(f"{path}  {path.stat().st_size / 1e6:.1f} MB")
    return path


f_pl = download(URL_PL)
for u in (URL_FORMAT, URL_USERINFO):
    download(u)

# what does the server say about the files?
requests.get(f"{BASE}/hfp_pl_csv/", timeout=30).json()

## 2. Look before you read

Never call `read_csv` on a file you have not looked at. Print the first bytes and lines **as they are**.

In [ ]:
raw = f_pl.read_bytes()
print(raw[:200])
print()
for line in raw.decode("utf-8-sig").splitlines()[:5]:
    print(line)

**Exercise:** write down the properties of this file that matter for reading it. Check each against the format description `data/original/geo0265/Datenformatbeschreibung.pdf`.

| Property | This file |
|---|---|
| encoding (what are the bytes `\xef\xbb\xbf` at the start?) | |
| line ending | |
| field separator | |
| decimal separator | |
| header line? | |
| text quoting | |
| columns and their meaning (format description, chapter 2) | |

## 3. Read the CSV

### 3.1 Read it with pandas
**Exercise:** read `f_pl` into a DataFrame `hfp` with the columns `id, name, east, north, h_nhn_m, g_mgal`. Follow the rule *identifiers as text*: the point ID must be a string. All other numeric columns must come out as `float64`.

In [ ]:
COLS = ["id", "name", "east", "north", "h_nhn_m", "g_mgal"]

# YOUR CODE HERE
hfp.head()

### 3.2 Check what you read

In [ ]:
print(hfp.dtypes, "\n")
print("rows:", len(hfp), "  unique IDs:", hfp["id"].nunique())
print("ID lengths:", hfp["id"].str.len().value_counts().to_dict())
print("missing values:", hfp.isna().sum().to_dict())
assert hfp["id"].map(type).eq(str).all(), "id must be text"
assert hfp[["east", "north", "h_nhn_m"]].dtypes.eq("float64").all(), "decimal comma not handled?"
# names that contain the separator ; – did the quoting work?
hfp[hfp["name"].str.contains(";")]

**Questions:**
1. How many benchmarks have no gravity value? Is that a reading error?
2. The IDs are ten digits long. Look at the first four digits of the IDs around Xanten and Kamp-Lintfort. Which other number system from this course do they remind you of? (Hint: geo0241, TK25 sheet 4304 Xanten.)

### 3.3 Statistics of the coordinates

In [ ]:
hfp[["east", "north", "h_nhn_m", "g_mgal"]].describe().round(3)

## 4. Which EPSG code?

The file says *east* and *north* in metres and nothing more. Old German documents call them *Rechtswert* and *Hochwert*, the terms of the **Gauss–Krüger** system. That does not mean the numbers are Gauss–Krüger coordinates.

### 4.1 Read the numbers
**Exercise:** answer from the statistics above, before you test anything:
* How many digits does `east` have before the decimal point? How many would a UTM easting have? A Gauss–Krüger *Rechtswert*?
* What do the first two digits of `east` mean?
* Write down your candidate EPSG codes (the slides of geo0220 have a list for NRW).

### 4.2 Test candidates
A wrong CRS usually does not raise an error, it just puts the points somewhere else. So transform a few points to geographic coordinates and see whether they land in NRW (roughly 5.8° E–9.5° E, 50.3° N–52.6° N).

`test_crs(epsg)` transforms the corners of the data (min/max east and north) with `always_xy=True`: the transformer then takes (east, north) and returns (λ, φ), whatever axis order the EPSG definition has.

**Exercise:** complete `test_crs` and run it for all candidates.

In [ ]:
NRW_BBOX = (5.8, 50.3, 9.5, 52.6)   # λ_min, φ_min, λ_max, φ_max


def test_crs(epsg, df=hfp):
    """Transform the data's min/max corners from epsg to ETRS89 geographic and check if they are in NRW."""
    crs = CRS.from_epsg(epsg)
    # YOUR CODE HERE
    lon, lat = np.round(lon, 3), np.round(lat, 3)
    inside = bool(np.all(np.isfinite(lon)) and NRW_BBOX[0] - 0.5 <= lon.min() and lon.max() <= NRW_BBOX[2] + 0.5
                  and NRW_BBOX[1] - 0.5 <= lat.min() and lat.max() <= NRW_BBOX[3] + 0.5)
    return {"epsg": epsg, "name": crs.name, "axes": "-".join(a.abbrev for a in crs.axis_info),
            "λ min/max": tuple(lon), "φ min/max": tuple(lat), "in NRW": inside}


pd.DataFrame([test_crs(e) for e in [4647, 5652, 25832, 31466, 31467]])

**Questions:**
1. Only one code puts the points into NRW. Which one, and what does `(zE-N)` in its name mean?
2. Why does EPSG:25832 fail although it is the same projection? What would you have to do to the `east` column to use EPSG:25832?
3. EPSG:5652 also lands in NRW. Look at the `axes` column: what is the difference between 4647 and 5652? `always_xy=True` hides it. Run the next cell, which uses the **axis order of the EPSG definition** (the default), and decide which of the two codes describes this file. (Hint: format description, chapter 2: which column comes first?)

In [ ]:
e, n = hfp.loc[0, ["east", "north"]]            # first benchmark, in the column order of the file
for epsg in (4647, 5652):
    t = Transformer.from_crs(epsg, 4258)         # authority axis order: output (φ, λ)
    print(epsg, "  (east, north) ->", np.round(t.transform(e, n), 4), "   (north, east) ->", np.round(t.transform(n, e), 4))

### 4.3 Compare with the metadata
Every official data set has a **metadata record** (ISO 19115/19139), here in the GDI-DE catalogue. It is meant for machines and for you:
* human-readable: the Open.NRW page of the record (link printed below), section *Additional information*,
* the format description PDF that came with the download: `Datenformatbeschreibung.pdf`.

The cell below reads the CRS and the bounding box from the XML record.

In [ ]:
print("Metadata (human-readable):", URL_META_HTML)
print("Metadata (ISO XML):       ", URL_META, "\n")
NS = {"gmd": "http://www.isotc211.org/2005/gmd", "gco": "http://www.isotc211.org/2005/gco",
      "gmx": "http://www.isotc211.org/2005/gmx"}
try:
    r = requests.get(URL_META, timeout=60)
    r.raise_for_status()
    root = ET.fromstring(r.content)
    meta_crs = [e.text for e in root.iterfind(".//gmd:referenceSystemIdentifier//gmd:code/*", NS)]
    meta_bbox = {tag: float(root.findtext(f".//gmd:EX_GeographicBoundingBox/gmd:{tag}/gco:Decimal", namespaces=NS))
                 for tag in ["westBoundLongitude", "southBoundLatitude", "eastBoundLongitude", "northBoundLatitude"]}
    print("title:", root.findtext(".//gmd:identificationInfo//gmd:title/gco:CharacterString", namespaces=NS))
    print("reference system(s):", meta_crs)
    print("bounding box:", meta_bbox)
    print("licence:", [e.text[:80] for e in root.iterfind(".//gmd:otherConstraints/gco:CharacterString", NS)
                       if "Lizenz" in (e.text or "") or "licen" in (e.text or "").lower()][:1])
except Exception as err:                     # catalogue blocked or down: read the record in the browser
    print("Could not read the metadata XML:", err)

**Exercise:** compare three sources and fill in the table.

| Source | What does it say about the CRS of east/north? | About the height? |
|---|---|---|
| the numbers (§4.2) | | – |
| ISO metadata record | | |
| `Datenformatbeschreibung.pdf`, chapter 2 | | |

**Questions:**
1. The format description says *"Ostwert (m im DHDN2016)"*. Look up DHDN and DHHN2016 in the slides of geo0220 and geo0260. Is there a "DHDN2016"? What was probably meant?
2. The height column is in DHHN2016. Which EPSG code does that vertical reference system have? (`CRS.from_epsg(7837)`). Why would EPSG:5555 be wrong for this file?
3. Do the corners of the data (§4.2, EPSG:4647) fit into the bounding box of the metadata?

In [ ]:
for e in (7837, 5555):
    print(e, CRS.from_epsg(e).name)

### 4.4 How precise are the positions?
Height benchmarks exist for their **heights** (levelled to millimetres). Their positions are a by-product. Count the decimals of `east` as written in the file (read it again as text).

In [ ]:
txt = pd.read_csv(f_pl, sep=";", header=None, names=COLS, encoding="utf-8-sig", dtype=str)
decimals = txt["east"].str.split(",").str[1].fillna("").str.len()
decimals.value_counts().sort_index().rename("number of benchmarks")

**Question:** Would you use these benchmarks as horizontal ground control points for a drone survey? For a height check?

## 5. GeoDataFrame and GeoPackage

The course standard for NRW is EPSG:25832 (no zone prefix). Build the points in the CRS of the file (EPSG:4647), then transform. Between 4647 and 25832 only the prefix changes, so the transformation must remove exactly 32 000 000 m from the easting: check it.

**Exercise:** create `gdf` (EPSG:4647) from `hfp`, transform it to `gdf25832`, and add the columns `e_25832`, `n_25832` with the new coordinates (rounded to mm).

In [ ]:
# YOUR CODE HERE
print("easting difference [m]:", (gdf25832["east"] - gdf25832["e_25832"]).agg(["min", "max"]).round(3).to_dict())
print("northing difference [m]:", round(float((gdf25832["north"] - gdf25832["n_25832"]).abs().max()), 3))
gdf25832.head(3)

Now save it. Keep the original coordinates under an honest name so nobody mistakes them for EPSG:25832 values later.

In [ ]:
GPKG = OUT / "hfp_nrw.gpkg"
layer = (gdf25832.rename(columns={"east": "e_4647", "north": "n_4647"})
         [["id", "name", "h_nhn_m", "g_mgal", "e_25832", "n_25832", "e_4647", "n_4647", "geometry"]])
layer.to_file(GPKG, layer="hfp_points", driver="GPKG")

# read it back: CRS, field types, number of features
check = gpd.read_file(GPKG, layer="hfp_points")
print(check.crs.to_epsg(), len(check), check.dtypes.to_dict())
assert check.crs.to_epsg() == 25832 and len(check) == len(hfp) and check["id"].map(type).eq(str).all()

Open `data/derived/geo0265/hfp_nrw.gpkg` in QGIS (or download it from the hub). QGIS should not ask for a CRS, and in *Layer Properties → Fields* the field `id` must be of type *String*/*Text*.

**Question:** The height `h_nhn_m` is stored as an attribute, the geometry is 2D. What would change if you stored 3D points (`POINT Z`)? Which (compound) CRS would the layer then need?

## 6. Benchmarks near the Green Fab Lab and the campus

### 6.1 Distances
**Exercise:** put the two places from `PLACES` into a GeoDataFrame `places` (EPSG:25832). Then build a table `near` of all benchmarks within **2 km** of each place with the columns `place, id, name, h_nhn_m, dist_m`, sorted by place and distance.

In [ ]:
RADIUS = 2000   # m

# YOUR CODE HERE
print(near.groupby("place").size().rename("benchmarks within 2 km"))
near.groupby("place").head(5)

An alternative for "the nearest *n*" without a radius is a spatial join: `gpd.sjoin_nearest(places, layer, distance_col="dist_m")` returns the single nearest benchmark per place. Try it.

In [ ]:
gpd.sjoin_nearest(places, layer, distance_col="dist_m")[["place", "id", "name", "h_nhn_m", "dist_m"]]

### 6.2 Map

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))
x0, y0 = places.geometry.x.mean(), places.geometry.y.mean()
win = layer.cx[x0 - 2500:x0 + 2500, y0 - 2500:y0 + 2500]
win.plot(ax=ax, color="#5D6D7E", markersize=12, label="height benchmark (HFP)")
near.plot(ax=ax, color="#1F6391", markersize=30, label=f"within {RADIUS / 1000:.0f} km")
places.plot(ax=ax, color="#A84300", marker="*", markersize=400, label="our places")
places.buffer(1000).boundary.plot(ax=ax, color="#A84300", linewidth=1, linestyle="--")
for _, p in places.iterrows():
    ax.annotate(p["place"], (p.geometry.x, p.geometry.y), xytext=(14, -4), textcoords="offset points",
                fontsize=12, color="#A84300", fontweight="bold")
label = near.query("place == 'Green Fab Lab'").nsmallest(4, "dist_m")         # four nearest to the GFL
for (_, r), off in zip(label.iterrows(), [(-150, -30), (12, 10), (-150, 12), (12, -30)]):
    ax.annotate(f"{r['id']}\n{r['h_nhn_m']:.3f} m", (r.geometry.x, r.geometry.y), xytext=off,
                textcoords="offset points", fontsize=11, color="#1F6391",
                arrowprops=dict(arrowstyle="-", color="#1F6391", lw=0.8))
ax.set_title("Height benchmarks around the campus Kamp-Lintfort (EPSG:25832, dashed: 1 km)", fontsize=14)
ax.set_xlabel("E [m]", fontsize=12); ax.set_ylabel("N [m]", fontsize=12)
ax.ticklabel_format(style="plain", useOffset=False); ax.tick_params(labelsize=11)
ax.legend(fontsize=12); ax.set_aspect("equal")
plt.show()

For an interactive map with a background, use `near.explore(column="place")` (needs `folium`). Data: Höhenfestpunkte NRW © Geobasis NRW, dl-de/zero-2-0.

**Questions:**
1. Which benchmark is closest to the Green Fab Lab? Read its name: what is it fixed to?
2. You want to check the NHN heights of the Emlid Reach RS2+ (slides: *Worked example: the Green Fab Lab*). Which two or three benchmarks would you visit, and what do you have to think about when you put an RTK pole on a bolt in a wall?
3. Write the IDs of the five nearest benchmarks with their leading digits. Which TK25 sheets are they on?

## 7. Stretch: heights change – the time series

The second file `hfp_plzf.csv` lists **every height ever determined** for each benchmark, with its date. Kamp-Lintfort lived on coal for a hundred years (Bergwerk Friedrich Heinrich, later Bergwerk West, closed 2012). Read the *Nutzerinformation.pdf* first: what may you conclude from such series and what not?

**Exercise:** read the time series (columns `id, date, h_nhn_m`; dates are `dd.mm.yyyy`), keep the benchmarks within 1 km of the Green Fab Lab, and plot height against time, one line per benchmark. Then compute for each benchmark the first and last date and the total change in cm.

In [ ]:
f_zf = download(URL_ZF)

# YOUR CODE HERE
summary.sort_values("change_cm")

**Questions:**
1. How much did the ground near the Green Fab Lab sink between the 1960s and today? How does that compare with the accuracy of an RTK height (slides)?
2. Several benchmarks have no measurement after the 1970s or 1980s. What may have happened to them? Would you rely on them in the field?
3. Check the whole file: `zf.groupby("id")["h_nhn_m"].agg(lambda h: h.max() - h.min()).nlargest(5)`. Can a benchmark really move 100 m? What does that tell you about using official data without checks?

## 8. Wrap-up
Answer in a few sentences:
1. Which EPSG code do the coordinates of the NRW height benchmarks have, how did you find out, and what is the difference to EPSG:25832?
2. Name one statement in the metadata/format description that is wrong or misleading, and how you noticed.
3. Which benchmark would you use to check the RTK heights at the Green Fab Lab, and why?

Homework: see the course plan.